# Notebook 2 — Create the Late-Delivery Label

## Purpose

Turn the order-level artifact from Notebook 1 into a labelled dataset for supervised learning.

## Prediction problem

At order-processing time, predict whether a successfully delivered order will arrive **after its promised calendar date**.

## Input artifact

`data/processed/orders_joined.parquet` from Notebook 1.

## Output artifact

`data/processed/orders_labelled.parquet`, containing one row per eligible delivered order.

## Unit of analysis

One order (`order_id`).

## Label definition

- `is_late = 1`: actual delivery **calendar date** is after the estimated delivery calendar date.
- `is_late = 0`: actual delivery is on or before the estimated calendar date.
- An order is label-eligible only when `order_status == "delivered"` and both dates exist.

Calendar dates are compared instead of raw timestamps. The estimated field is stored at midnight, so a raw timestamp comparison would incorrectly mark an order delivered later on the promised day as late.

## Leakage warning

`order_delivered_customer_date`, `delivery_delay_days`, reviews, and other post-outcome information can be used to **construct or audit** the label, but must never become model inputs.

## 1. Import the required libraries

This notebook reads and writes Parquet files, validates the table, and records file hashes. It does not connect to PostgreSQL because Notebook 1 already created the reproducible input artifact.

In [1]:
from hashlib import sha256
from pathlib import Path

import numpy as np
import pandas as pd

print(f"NumPy version: {np.__version__}")
print(f"pandas version: {pd.__version__}")
print("Imports completed successfully.")

NumPy version: 2.5.2
pandas version: 3.0.5
Imports completed successfully.


## 2. Resolve project paths

The path logic works whether Jupyter starts from the repository root or from the `notebooks` directory. No machine-specific absolute path is stored in the notebook.

In [2]:
WORKING_DIR = Path.cwd().resolve()
PROJECT_ROOT = WORKING_DIR.parent if WORKING_DIR.name == "notebooks" else WORKING_DIR

assert (PROJECT_ROOT / "docker-compose.yml").exists(), (
    "Project root could not be identified. Run this notebook from the repository "
    "root or its notebooks directory."
)

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
INPUT_PATH = PROCESSED_DATA_DIR / "orders_joined.parquet"
OUTPUT_PATH = PROCESSED_DATA_DIR / "orders_labelled.parquet"

assert INPUT_PATH.exists(), (
    f"Missing input artifact: {INPUT_PATH}. Run Notebook 1 first."
)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Input artifact: {INPUT_PATH}")
print(f"Output artifact: {OUTPUT_PATH}")

Project root: D:\Documents\mlops-olist
Input artifact: D:\Documents\mlops-olist\data\processed\orders_joined.parquet
Output artifact: D:\Documents\mlops-olist\data\processed\orders_labelled.parquet


## 3. Load and validate the Notebook 1 artifact

The checks establish the contract between Notebook 1 and Notebook 2: 99,441 order rows, 65 columns, one unique non-null `order_id` per row, and the fields needed to construct the label.

In [3]:
orders = pd.read_parquet(INPUT_PATH)

required_columns = {
    "order_id",
    "order_status",
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
}

missing_required_columns = sorted(required_columns - set(orders.columns))

print(f"Input shape: {orders.shape}")
print(f"Unique order IDs: {orders['order_id'].nunique()}")
print(f"Duplicated order IDs: {orders['order_id'].duplicated().sum()}")
print(f"Missing order IDs: {orders['order_id'].isna().sum()}")
print(f"Missing required columns: {missing_required_columns}")

assert orders.shape == (99_441, 65), "Unexpected Notebook 1 artifact shape."
assert not missing_required_columns, "Required columns are missing."
assert orders["order_id"].notna().all(), "order_id contains missing values."
assert orders["order_id"].is_unique, "order_id must be unique."

Input shape: (99441, 65)
Unique order IDs: 99441
Duplicated order IDs: 0
Missing order IDs: 0
Missing required columns: []


## 4. Parse and inspect the label dates

Explicit conversion prevents string comparisons. Invalid values become missing (`NaT`) and are counted rather than silently labelled.

In [4]:
date_columns = [
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for column in date_columns:
    orders[column] = pd.to_datetime(orders[column], errors="coerce")

date_audit = pd.DataFrame(
    {
        "missing_count": orders[date_columns].isna().sum(),
        "missing_percent": orders[date_columns].isna().mean().mul(100).round(4),
        "minimum": orders[date_columns].min(),
        "maximum": orders[date_columns].max(),
    }
)

display(date_audit)

,missing_count,missing_percent,minimum,maximum
order_purchase_timestamp,0,0.0000,2016-09-04 21:15:19,2018-10-17 17:30:18
order_delivered_customer_date,2965,2.9817,2016-10-11 13:46:32,2018-10-17 13:22:46
order_estimated_delivery_date,0,0.0000,2016-09-30 00:00:00,2018-11-12 00:00:00


## 5. Define label eligibility

Only completed deliveries have an observed delivery outcome. Non-delivered orders and rows missing either comparison date are excluded from the labelled artifact, but their exclusion reasons are audited below.

In [5]:
is_delivered_status = orders["order_status"].eq("delivered")
has_actual_delivery_date = orders["order_delivered_customer_date"].notna()
has_estimated_delivery_date = orders["order_estimated_delivery_date"].notna()

label_eligible = (
    is_delivered_status
    & has_actual_delivery_date
    & has_estimated_delivery_date
)

exclusion_audit = pd.DataFrame(
    {
        "rule": [
            "All source orders",
            "Non-delivered status",
            "Delivered but actual date missing",
            "Delivered but estimated date missing",
            "Eligible delivered orders",
        ],
        "order_count": [
            len(orders),
            (~is_delivered_status).sum(),
            (is_delivered_status & ~has_actual_delivery_date).sum(),
            (is_delivered_status & ~has_estimated_delivery_date).sum(),
            label_eligible.sum(),
        ],
    }
)

display(exclusion_audit)

excluded_status_counts = (
    orders.loc[~label_eligible, "order_status"]
    .value_counts(dropna=False)
    .rename_axis("order_status")
    .reset_index(name="order_count")
)

print("Excluded orders by status:")
display(excluded_status_counts)

assert label_eligible.any(), "No orders are eligible for labelling."

,rule,order_count
0,All source orders,99441
1,Non-delivered status,2963
2,Delivered but actual date missing,8
3,Delivered but estimated date missing,0
4,Eligible delivered orders,96470


Excluded orders by status:


,order_status,order_count
0,shipped,1107
1,canceled,625
2,unavailable,609
3,invoiced,314
4,processing,301
5,delivered,8
6,created,5
7,approved,2


## 6. Create the label using calendar dates

`delivery_delay_days` is the signed difference between actual and estimated dates:

- positive: delivered late;
- zero: delivered on the promised date;
- negative: delivered early.

It explains the label but is an outcome-derived leakage column, not a future model feature.

In [6]:
labelled_orders = orders.loc[label_eligible].copy()

actual_delivery_date = labelled_orders[
    "order_delivered_customer_date"
].dt.normalize()

estimated_delivery_date = labelled_orders[
    "order_estimated_delivery_date"
].dt.normalize()

labelled_orders["delivery_delay_days"] = (
    actual_delivery_date - estimated_delivery_date
).dt.days.astype("int16")

labelled_orders["is_late"] = (
    labelled_orders["delivery_delay_days"] > 0
).astype("int8")

print(f"Labelled shape: {labelled_orders.shape}")
print(f"Missing labels: {labelled_orders['is_late'].isna().sum()}")
print(f"Valid label values: {sorted(labelled_orders['is_late'].unique())}")

assert labelled_orders["is_late"].notna().all()
assert set(labelled_orders["is_late"].unique()) <= {0, 1}
assert (
    labelled_orders["is_late"]
    == (labelled_orders["delivery_delay_days"] > 0).astype("int8")
).all()

Labelled shape: (96470, 67)
Missing labels: 0
Valid label values: [np.int8(0), np.int8(1)]


## 7. Inspect class balance

Accuracy alone can be misleading if late deliveries are the minority. The class distribution determines whether later evaluation should emphasize metrics such as recall, precision, F1, PR-AUC, or ROC-AUC.

In [7]:
class_distribution = (
    labelled_orders["is_late"]
    .value_counts()
    .sort_index()
    .rename_axis("is_late")
    .reset_index(name="order_count")
)

class_distribution["label_name"] = class_distribution["is_late"].map(
    {0: "on_time_or_early", 1: "late"}
)
class_distribution["percent"] = (
    class_distribution["order_count"]
    .div(len(labelled_orders))
    .mul(100)
    .round(4)
)

class_distribution = class_distribution[
    ["is_late", "label_name", "order_count", "percent"]
]

display(class_distribution)

late_rate = labelled_orders["is_late"].mean()
majority_baseline_accuracy = labelled_orders["is_late"].value_counts(normalize=True).max()

print(f"Late-delivery rate: {late_rate:.4%}")
print(f"Majority-class accuracy baseline: {majority_baseline_accuracy:.4%}")

,is_late,label_name,order_count,percent
0,0,on_time_or_early,89936,93.2269
1,1,late,6534,6.7731


Late-delivery rate: 6.7731%
Majority-class accuracy baseline: 93.2269%


## 8. Manually validate real orders

These examples make the business rule visible. Boundary cases are especially useful: deliveries exactly on the estimated date must have label 0; deliveries one day after must have label 1.

In [8]:
validation_columns = [
    "order_id",
    "order_status",
    "order_purchase_timestamp",
    "order_estimated_delivery_date",
    "order_delivered_customer_date",
    "delivery_delay_days",
    "is_late",
]

boundary_examples = (
    labelled_orders.loc[
        labelled_orders["delivery_delay_days"].isin([-1, 0, 1]),
        validation_columns,
    ]
    .sort_values(["delivery_delay_days", "order_id"])
    .groupby("delivery_delay_days", group_keys=False)
    .head(3)
)

display(boundary_examples)

same_day_rows = labelled_orders["delivery_delay_days"].eq(0)
one_day_late_rows = labelled_orders["delivery_delay_days"].eq(1)

assert labelled_orders.loc[same_day_rows, "is_late"].eq(0).all()
assert labelled_orders.loc[one_day_late_rows, "is_late"].eq(1).all()

,order_id,order_status,order_purchase_timestamp,order_estimated_delivery_date,order_delivered_customer_date,delivery_delay_days,is_late
48,001e7cf2ad6bef3ade12ebc56ceaf0f3,delivered,2018-05-19 10:29:23,2018-06-05,2018-06-04 18:08:23,-1,0
96,003cc6161d7a2593f2525cce0c330d32,delivered,2018-08-03 19:33:35,2018-08-08,2018-08-07 11:32:11,-1,0
100,003edccf16bc5ec447f592913b3df2b4,delivered,2018-07-07 09:02:51,2018-08-03,2018-08-02 23:07:33,-1,0
8,0005a1a1728c9d785b8e2b08b904576c,delivered,2018-03-19 18:40:33,2018-03-29,2018-03-29 18:17:31,0,0
11,00063b381e2406b52ad429470734ebd5,delivered,2018-07-27 17:21:27,2018-08-07,2018-08-07 13:56:52,0,0
91,00378c6c981f234634c0b9d6128df6dd,delivered,2018-02-02 19:39:46,2018-02-26,2018-02-26 21:36:00,0,0
125,00526a9d4ebde463baee25f386963ddc,delivered,2018-08-07 22:03:44,2018-08-15,2018-08-16 19:58:24,1,1
478,0136bcc7370d7fdf44bd916a6dd583c6,delivered,2018-05-20 19:31:30,2018-06-05,2018-06-06 18:53:26,1,1
716,01d95eed9163a292a6788ddac08bcf31,delivered,2017-11-08 09:54:42,2017-12-04,2017-12-05 00:26:03,1,1


## 9. Audit implausible chronology

We do not silently repair suspicious records. Counts are reported so later notebooks can make explicit filtering decisions if necessary.

In [9]:
chronology_audit = pd.DataFrame(
    {
        "condition": [
            "Delivery before purchase",
            "Estimated delivery before purchase",
            "Delay below -100 days",
            "Delay above 100 days",
        ],
        "order_count": [
            (
                labelled_orders["order_delivered_customer_date"]
                < labelled_orders["order_purchase_timestamp"]
            ).sum(),
            (
                labelled_orders["order_estimated_delivery_date"]
                < labelled_orders["order_purchase_timestamp"].dt.normalize()
            ).sum(),
            labelled_orders["delivery_delay_days"].lt(-100).sum(),
            labelled_orders["delivery_delay_days"].gt(100).sum(),
        ],
    }
)

display(chronology_audit)

,condition,order_count
0,Delivery before purchase,0
1,Estimated delivery before purchase,0
2,Delay below -100 days,5
3,Delay above 100 days,39


## 10. Validate the final labelled table

The table must remain at one row per eligible order. Sorting by `order_id` makes row order deterministic for downstream notebooks.

In [10]:
labelled_orders = labelled_orders.sort_values("order_id").reset_index(drop=True)

final_checks = {
    "source_rows": len(orders),
    "eligible_rows": len(labelled_orders),
    "excluded_rows": len(orders) - len(labelled_orders),
    "unique_order_ids": labelled_orders["order_id"].nunique(),
    "duplicated_order_ids": labelled_orders["order_id"].duplicated().sum(),
    "missing_order_ids": labelled_orders["order_id"].isna().sum(),
    "missing_labels": labelled_orders["is_late"].isna().sum(),
}

for check_name, value in final_checks.items():
    print(f"{check_name}: {value}")

assert labelled_orders["order_id"].is_unique
assert labelled_orders["order_id"].notna().all()
assert labelled_orders["is_late"].notna().all()
assert len(labelled_orders) == int(label_eligible.sum())
assert labelled_orders["order_id"].is_monotonic_increasing

source_rows: 99441
eligible_rows: 96470
excluded_rows: 2971
unique_order_ids: 96470
duplicated_order_ids: 0
missing_order_ids: 0
missing_labels: 0


## 11. Save and reload the labelled artifact

The reload check proves the next notebook can consume the saved file without rerunning this notebook. The SHA-256 hash records the exact local artifact version.

In [11]:
labelled_orders.to_parquet(OUTPUT_PATH, index=False)
reloaded_orders = pd.read_parquet(OUTPUT_PATH)

pd.testing.assert_frame_equal(
    labelled_orders,
    reloaded_orders,
    check_dtype=True,
    check_like=False,
)

artifact_hash = sha256(OUTPUT_PATH.read_bytes()).hexdigest().upper()
artifact_size_bytes = OUTPUT_PATH.stat().st_size

print(f"Saved artifact: {OUTPUT_PATH}")
print(f"Artifact rows: {len(reloaded_orders)}")
print(f"Artifact columns: {len(reloaded_orders.columns)}")
print(f"Artifact size: {artifact_size_bytes:,} bytes")
print(f"SHA-256: {artifact_hash}")
print("Parquet round-trip validation passed.")

Saved artifact: D:\Documents\mlops-olist\data\processed\orders_labelled.parquet
Artifact rows: 96470
Artifact columns: 67
Artifact size: 21,414,799 bytes
SHA-256: F52F44DA5BEA0CFC3B279BE488CBAF0B61F9853F838A2114BF1750FAF5C208AC
Parquet round-trip validation passed.


## 12. Completion summary and handoff

Notebook 2 is complete when:

- the label rule uses calendar dates;
- only eligible delivered orders are labelled;
- labels contain only 0 and 1 with no missing values;
- order IDs remain unique;
- real boundary examples validate the rule;
- the labelled Parquet artifact survives an exact save/reload check.

Notebook 3 will read `orders_labelled.parquet` and split it into train, validation, and test sets **before detailed EDA**. The split decision must consider time, class balance, and leakage. Outcome-derived and post-delivery columns will not be used as model inputs.

In [12]:
late_count = int(labelled_orders["is_late"].sum())
on_time_count = int((labelled_orders["is_late"] == 0).sum())

print("NOTEBOOK 2 FINAL RESULT")
print(f"Source orders: {len(orders)}")
print(f"Labelled orders: {len(labelled_orders)}")
print(f"Excluded orders: {len(orders) - len(labelled_orders)}")
print(f"On-time/early orders: {on_time_count}")
print(f"Late orders: {late_count}")
print(f"Late rate: {late_rate:.4%}")
print(f"Duplicate order IDs: {labelled_orders['order_id'].duplicated().sum()}")
print(f"Output shape: {labelled_orders.shape}")
print(f"Output SHA-256: {artifact_hash}")
print("Ready for Notebook 3: YES")

NOTEBOOK 2 FINAL RESULT
Source orders: 99441
Labelled orders: 96470
Excluded orders: 2971
On-time/early orders: 89936
Late orders: 6534
Late rate: 6.7731%
Duplicate order IDs: 0
Output shape: (96470, 67)
Output SHA-256: F52F44DA5BEA0CFC3B279BE488CBAF0B61F9853F838A2114BF1750FAF5C208AC
Ready for Notebook 3: YES
